In [1]:
import pandas  as pd

In [2]:
df = pd.read_csv('../临时文件/大R留存.csv',index_col=False)

In [4]:
df=df[['用户唯一id','事件发生时间']]

In [5]:
df['时间']= pd.to_datetime(df['事件发生时间'],unit='ms')

C:\Users\admin\AppData\Local\Temp\ipykernel_21524\2292237435.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['时间']= pd.to_datetime(df['事件发生时间'],unit='ms')


In [8]:
# 排序，找出首次登录
df['rank_login'] = df.groupby(by='用户唯一id')['时间'].rank(ascending=True,method='first')

C:\Users\admin\AppData\Local\Temp\ipykernel_21524\2651029525.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rank_login'] = df.groupby(by='用户唯一id')['时间'].rank(ascending=True,method='first')


In [12]:
df.sort_values(by=['用户唯一id','rank_login'],ascending=[True,True],inplace=True)

C:\Users\admin\AppData\Local\Temp\ipykernel_21524\1298682937.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_values(by=['用户唯一id','rank_login'],ascending=[True,True],inplace=True)


In [13]:
df

,用户唯一id,事件发生时间,时间,rank_login
3908,10R8X699GXMV3rvfkXcVMC,1728760783503,2024-10-12 19:19:43.503,1.0
1809,10R8X699GXMV3rvfkXcVMC,1728765125495,2024-10-12 20:32:05.495,2.0
1981,10R8X699GXMV3rvfkXcVMC,1728766017366,2024-10-12 20:46:57.366,3.0
3311,10R8X699GXMV3rvfkXcVMC,1728795001795,2024-10-13 04:50:01.795,4.0
1500,10R8X699GXMV3rvfkXcVMC,1728810829047,2024-10-13 09:13:49.047,5.0
...,...,...,...,...
1020,yTbgf9zNVkZfXUlSz65it,1729068957321,2024-10-16 08:55:57.321,11.0
2072,yTbgf9zNVkZfXUlSz65it,1729113096350,2024-10-16 21:11:36.350,12.0
2070,yTbgf9zNVkZfXUlSz65it,1729148437346,2024-10-17 07:00:37.346,13.0
3281,yTbgf9zNVkZfXUlSz65it,1729165999296,2024-10-17 11:53:19.296,14.0


In [15]:
df1=df.query('rank_login==1')[['用户唯一id','时间']]

In [17]:
df1.columns = ['用户唯一id','first_login']

In [19]:
res=df.merge(df1,how='left',on='用户唯一id')

In [22]:
res['间隔天数'] =(res['时间'] - res['first_login']).dt.days

In [23]:
res

,用户唯一id,事件发生时间,时间,rank_login,first_login,间隔天数
0,10R8X699GXMV3rvfkXcVMC,1728760783503,2024-10-12 19:19:43.503,1.0,2024-10-12 19:19:43.503,0
1,10R8X699GXMV3rvfkXcVMC,1728765125495,2024-10-12 20:32:05.495,2.0,2024-10-12 19:19:43.503,0
2,10R8X699GXMV3rvfkXcVMC,1728766017366,2024-10-12 20:46:57.366,3.0,2024-10-12 19:19:43.503,0
3,10R8X699GXMV3rvfkXcVMC,1728795001795,2024-10-13 04:50:01.795,4.0,2024-10-12 19:19:43.503,0
4,10R8X699GXMV3rvfkXcVMC,1728810829047,2024-10-13 09:13:49.047,5.0,2024-10-12 19:19:43.503,0
...,...,...,...,...,...,...
4300,yTbgf9zNVkZfXUlSz65it,1729068957321,2024-10-16 08:55:57.321,11.0,2024-10-13 11:53:06.755,2
4301,yTbgf9zNVkZfXUlSz65it,1729113096350,2024-10-16 21:11:36.350,12.0,2024-10-13 11:53:06.755,3
4302,yTbgf9zNVkZfXUlSz65it,1729148437346,2024-10-17 07:00:37.346,13.0,2024-10-13 11:53:06.755,3
4303,yTbgf9zNVkZfXUlSz65it,1729165999296,2024-10-17 11:53:19.296,14.0,2024-10-13 11:53:06.755,4


In [31]:
# 分日期查看 10月份之前用户
res.query('first_login<"2024-10-01"').pivot_table(values="用户唯一id",# 计算字段
                      # index="name",# 分组的行
                      columns="间隔天数",# 分组的列
                      aggfunc="nunique", # 聚合方式
                      # fill_value=0, # 对缺失值的填充，默认为nan
                      # margins=True, # 是否启用总计
                      # margins_name='总计' # 总计的名称
                     ).to_excel('../临时文件/大R留存1.xlsx',index=False)

In [28]:
res['日期'] = res['first_login'].dt.strftime('%Y-%m-%d')

In [32]:
# 分日期查看 10月份之后用户
res.query('first_login>"2024-10-01"').pivot_table(values="用户唯一id",# 计算字段
                      index="日期",# 分组的行
                      columns="间隔天数",# 分组的列
                      aggfunc="nunique", # 聚合方式
                      # fill_value=0, # 对缺失值的填充，默认为nan
                      # margins=True, # 是否启用总计
                      # margins_name='总计' # 总计的名称
                     ).to_excel('../临时文件/大R留存.xlsx')